In [0]:
%sql
-- Convert types and identity invalid records
CREATE OR REPLACE TEMP VIEW sales_checked AS
with PARSED AS (
    SELECT
        order_id,
        line_id,
        product,
        category,
        order_date AS raw_order_date,
        quantity AS raw_quantity,
        unit_price AS raw_unit_price,
        TRY_CAST(order_date AS DATE) AS order_date,
        TRY_CAST(quantity AS INT) AS quantity,
        TRY_CAST(unit_price AS DECIMAL(10,2)) AS unit_price,
        ingested_at,
        source_name
    FROM workspace.portfolio_bronze.sales_raw
)
SELECT
  *,
  CASE
    WHEN order_id IS NULL OR TRIM(order_id) = ''
      THEN 'Missing order ID'
    WHEN line_id IS NULL OR TRIM(line_id) = ''
      THEN 'Missing line ID'
    WHEN product IS NULL OR TRIM(product) = ''
      THEN 'Missing product'
    WHEN category IS NULL OR TRIM(category) = ''
      THEN 'Missing category'
    WHEN order_date IS NULL
      THEN 'Invalid order date'
    WHEN quantity IS NULL OR quantity <= 0
      THEN 'Quantity must be a positive integer'
    WHEN unit_price IS NULL OR unit_price < 0
      THEN 'Price must be a non-negative number'
    ELSE NULL
  END AS rejection_reason
FROM parsed;



In [0]:
%sql
-- Save the rejected records
CREATE OR REPLACE TABLE workspace.portfolio_silver.sales_rejected
USING DELTA
AS
SELECT *
FROM sales_checked
WHERE rejection_reason IS NOT NULL;

-- This keeps the original problematic values alongside the reason each record failed.

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Save clean records and remove duplicates
CREATE OR REPLACE TABLE workspace.portfolio_silver.sales_clean
USING DELTA
AS
SELECT DISTINCT
  order_id,
  line_id,
  order_date,
  product,
  category,
  quantity,
  unit_price,
  CAST(quantity * unit_price AS DECIMAL(18, 2)) AS line_revenue
FROM sales_checked
WHERE rejection_reason IS NULL;

-- DISTINCT removes repeated rows with identical selecte values. 
-- CREATE OR REPLACE rebuilds these derived Silver tables when rerun. Bronze remains our original source.

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Verify the results
SELECT 'Bronze records' AS metric, COUNT(*) AS record_count
FROM workspace.portfolio_bronze.sales_raw

UNION ALL

SELECT 'Silver clean records', COUNT(*)
FROM workspace.portfolio_silver.sales_clean

UNION ALL

SELECT 'Rejected records', COUNT(*)
FROM workspace.portfolio_silver.sales_rejected

-- Expected counts: Bronze 10, Silver clean 7, rejected 2. The remaining record was the duplicate.

metric,record_count
Bronze records,10
Silver clean records,7
Rejected records,2


In [0]:
%sql
-- Inspect the rejected records in another cell
SELECT order_id, raw_order_date, raw_quantity, rejection_reason
FROM workspace.portfolio_silver.sales_rejected
ORDER BY order_id;

order_id,raw_order_date,raw_quantity,rejection_reason
1007,not-a-date,1,Invalid order date
1008,2026-09-03,abc,Quantity must be a positive integer
